<a href="https://colab.research.google.com/github/kaii0802/CS-5530---Kai-Son/blob/main/q1_frailty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q1 - Grip strength and frailty

Three-stage workflow: **ingest -> process -> analyze**.

| Stage | Input | Output |
|---|---|---|
| 1. Ingest | table from the assignment | `data/raw/frailty_raw.csv` -> pandas DataFrame |
| 2. Process | raw DataFrame | `data/processed/frailty_processed.csv` |
| 3. Analyze | processed DataFrame | `reports/findings.md` |

This notebook lives in `q1_frailty/src/`; every path below is resolved relative to the `q1_frailty/` folder.

In [4]:
from pathlib import Path

import pandas as pd
import numpy as np
from scipy import stats

# The notebook lives in <question>/src/, so the question folder is its parent.
ROOT = Path("/content/CS-5530---Kai-Son/Assignments/Assignment 1/q1_frailty")
RAW_CSV = ROOT / "data" / "raw" / "frailty_raw.csv"
PROCESSED_CSV = ROOT / "data" / "processed" / "frailty_processed.csv"
REPORT = ROOT / "reports" / "findings.md"
for folder in (RAW_CSV.parent, PROCESSED_CSV.parent, REPORT.parent):
    folder.mkdir(parents=True, exist_ok=True)

## Stage 1 - Ingest

Save the raw data exactly as given (Height = inches, Weight = pounds, Age = years, Grip strength = kg)
to a CSV file, then read that CSV back into a pandas DataFrame.

In [5]:
RAW_ROWS = [
    (65.8, 112, 30, 30, "N"),
    (71.5, 136, 19, 31, "N"),
    (69.4, 153, 45, 29, "N"),
    (68.2, 142, 22, 28, "Y"),
    (67.8, 144, 29, 24, "Y"),
    (68.7, 123, 50, 26, "N"),
    (69.8, 141, 51, 22, "Y"),
    (70.1, 136, 23, 20, "Y"),
    (67.9, 112, 17, 19, "N"),
    (66.8, 120, 39, 31, "N"),
]
COLUMNS = ["Height_in", "Weight_lb", "Age_yr", "Grip_kg", "Frailty"]

pd.DataFrame(RAW_ROWS, columns=COLUMNS).to_csv(RAW_CSV, index=False)   # save raw data
raw = pd.read_csv(RAW_CSV)                                             # read it back

assert list(raw.columns) == COLUMNS
assert raw.notna().all().all(), "Raw data contains missing values"
raw

,Height_in,Weight_lb,Age_yr,Grip_kg,Frailty
0,65.8,112,30,30,N
1,71.5,136,19,31,N
2,69.4,153,45,29,N
3,68.2,142,22,28,Y
4,67.8,144,29,24,Y
5,68.7,123,50,26,N
6,69.8,141,51,22,Y
7,70.1,136,23,20,Y
8,67.9,112,17,19,N
9,66.8,120,39,31,N


## Stage 2 - Process

### (a) Unit standardization

In [6]:
df = raw.copy()
df["Height_m"] = df["Height_in"] * 0.0254
df["Weight_kg"] = df["Weight_lb"] * 0.45359237
df[["Height_in", "Height_m", "Weight_lb", "Weight_kg"]]

,Height_in,Height_m,Weight_lb,Weight_kg
0,65.8,1.67132,112,50.802345
1,71.5,1.81610,136,61.688562
2,69.4,1.76276,153,69.399633
3,68.2,1.73228,142,64.410117
4,67.8,1.72212,144,65.317301
5,68.7,1.74498,123,55.791862
6,69.8,1.77292,141,63.956524
7,70.1,1.78054,136,61.688562
8,67.9,1.72466,112,50.802345
9,66.8,1.69672,120,54.431084


### (b) Feature engineering

`BMI = Weight_kg / Height_m**2` (2 decimals). `AgeGroup` uses half-open bins so each age falls in exactly one
group: `<30` = [0, 30), `30–45` = [30, 46), `46–60` = [46, 61), `>60` = [61, inf).

In [7]:
AGE_LABELS = ["<30", "30–45", "46–60", ">60"]

df["BMI"] = (df["Weight_kg"] / (df["Height_m"] ** 2)).round(2)
df["AgeGroup"] = pd.cut(df["Age_yr"], bins=[0, 30, 46, 61, float("inf")], labels=AGE_LABELS, right=False)
df[["Age_yr", "AgeGroup", "BMI"]]

,Age_yr,AgeGroup,BMI
0,30,30–45,18.19
1,19,<30,18.70
2,45,30–45,22.33
3,22,<30,21.46
4,29,<30,22.02
5,50,46–60,18.32
6,51,46–60,20.35
7,23,<30,19.46
8,17,<30,17.08
9,39,30–45,18.91


### (c) Categorical -> numeric encoding


- `Frailty_binary`: Y -> 1, N -> 0 (`int8`).
- `AgeGroup` is one-hot encoded into four columns; all four are kept
even though nobody is older than 60 (so `AgeGroup_>60` is all zeros).

In [8]:
df["Frailty_binary"] = df["Frailty"].map({"Y": 1, "N": 0}).astype("int8")

dummies = pd.get_dummies(df["AgeGroup"], prefix="AgeGroup", dtype="int8")
dummies = dummies.reindex(columns=[f"AgeGroup_{g}" for g in AGE_LABELS], fill_value=0)
df = pd.concat([df, dummies], axis=1)

df.to_csv(PROCESSED_CSV, index=False)
print("Saved:", PROCESSED_CSV.relative_to(ROOT))
df

Saved: data/processed/frailty_processed.csv


,Height_in,Weight_lb,Age_yr,Grip_kg,Frailty,Height_m,Weight_kg,BMI,AgeGroup,Frailty_binary,AgeGroup_<30,AgeGroup_30–45,AgeGroup_46–60,AgeGroup_>60
0,65.8,112,30,30,N,1.67132,50.802345,18.19,30–45,0,0,1,0,0
1,71.5,136,19,31,N,1.81610,61.688562,18.70,<30,0,1,0,0,0
2,69.4,153,45,29,N,1.76276,69.399633,22.33,30–45,0,0,1,0,0
3,68.2,142,22,28,Y,1.73228,64.410117,21.46,<30,1,1,0,0,0
4,67.8,144,29,24,Y,1.72212,65.317301,22.02,<30,1,1,0,0,0
5,68.7,123,50,26,N,1.74498,55.791862,18.32,46–60,0,0,0,1,0
6,69.8,141,51,22,Y,1.77292,63.956524,20.35,46–60,1,0,0,1,0
7,70.1,136,23,20,Y,1.78054,61.688562,19.46,<30,1,1,0,0,0
8,67.9,112,17,19,N,1.72466,50.802345,17.08,<30,0,1,0,0,0
9,66.8,120,39,31,N,1.69672,54.431084,18.91,30–45,0,0,1,0,0


## Stage 3 - Analyze

### (d-I) Summary table: mean / median / std of the numeric columns

In [9]:
NUMERIC_COLS = ["Height_in", "Weight_lb", "Age_yr", "Grip_kg", "Height_m", "Weight_kg", "BMI", "Frailty_binary"]

summary_table = df[NUMERIC_COLS].agg(["mean", "median", "std"]).T.round(2)
summary_table.index.name = "Variable"
summary_table

,mean,median,std
Variable,,,
Height_in,68.60,68.45,1.67
Weight_lb,131.90,136.00,14.23
Age_yr,32.50,29.50,12.86
Grip_kg,26.00,27.00,4.52
Height_m,1.74,1.74,0.04
Weight_kg,59.83,61.69,6.46
BMI,19.68,19.19,1.78
Frailty_binary,0.40,0.00,0.52


### (d-II) Strength <-> frailty: correlation between `Grip_kg` and `Frailty_binary`

Pearson's r with a 0/1 variable is the point-biserial correlation. Spearman's rho is a rank-based check.

In [10]:
correlation = df["Grip_kg"].corr(df["Frailty_binary"])
print(f"Correlation coefficient between Grip Strength and Frailty: {correlation:.2f}")

by_group = df.groupby("Frailty")["Grip_kg"].agg(["count", "mean", "median", "std"]).round(2)
by_group

Correlation coefficient between Grip Strength and Frailty: -0.48


,count,mean,median,std
Frailty,,,,
N,6,27.67,29.5,4.63
Y,4,23.50,23.0,3.42


### Write `reports/findings.md`

The report contains the summary table, the processed data, the correlation, and a short interpretation.

In [11]:
# AgeGroup counts (for the report)
age_counts = df["AgeGroup"].value_counts().reindex(["<30", "30-45", "46-60", ">60"]).fillna(0).astype(int)


direction = "Lower" if correlation < 0 else "Higher"
mean_frail = by_group.loc["Y", "mean"]
mean_not_frail = by_group.loc["N", "mean"]
n_frail = int(df["Frailty_binary"].sum())
n_total = len(df)

# Write the full report
with open(ROOT / "reports" / "findings.md", "w") as f:
    f.write("# Q1 - Frailty Study: Findings\n\n")
    f.write(f"Sample: n = {n_total} female participants ({n_frail} frail, {n_total - n_frail} not frail).\n\n")

    f.write("## 1. Summary statistics (numeric columns)\n\n")
    f.write("Units: Height_in = inches, Weight_lb = pounds, Age_yr = years, Grip_kg = kg, "
            "Height_m = metres, Weight_kg = kg, BMI = kg/m2.\n\n")
    f.write(summary_table.to_markdown())
    f.write("\n\n")

    f.write("## 2. Processed data\n\n")
    f.write(df[["Height_in", "Weight_lb", "Age_yr", "Grip_kg", "Frailty",
                "Height_m", "Weight_kg", "BMI", "AgeGroup", "Frailty_binary"]].to_markdown())
    f.write("\n\n")
    f.write("AgeGroup counts: " + ", ".join(f"{k} = {v}" for k, v in age_counts.items()) + ". ")
    f.write("No participant is older than 60, so the AgeGroup_>60 one-hot column is all zeros ")

    f.write("## 3. Relationship between grip strength and frailty\n\n")
    f.write(by_group.to_markdown())
    f.write(f"\n\nCorrelation coefficient between Grip_kg and Frailty_binary: **{correlation:.2f}**\n\n")
    f.write(f"**Interpretation.** The correlation is {direction.lower()}: participants with higher grip "
            f"strength tend to be coded 0 (not frail), and those with lower grip strength tend to be "
            f"coded 1 (frail). The frail group averages {mean_frail:.1f} kg versus {mean_not_frail:.1f} kg "
            f"for the non-frail group. With only n = {n_total}, this is a plausible signal from a small "
            f"sample, not a firm conclusion.\n\n")

    f.write("## Conclusion\n\n")
    f.write(f"{direction} grip strength goes with more frailty in this small sample "
            f"(correlation = {correlation:.2f}), matching the pattern the assignment describes. "
            f"A larger sample would be needed to confirm it.\n")

print("\nfindings.md saved")
print("--- Stage 3: Analyze Complete ---")


findings.md saved
--- Stage 3: Analyze Complete ---
